# Setup

Install InterProt, load ESM and SAE

In [1]:
%%capture
!pip install git+https://github.com/etowahadams/interprot.git

In [2]:
#Uninstalled and reinstalled to fix numpy binary incompatibility issues
!pip uninstall -y numpy pandas polars interprot
!pip install numpy pandas polars git+https://github.com/etowahadams/interprot.git

import csv
import os
import torch
import warnings

import numpy as np
import pandas as pd
import polars as pl

from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.metrics import accuracy_score
from tqdm import tqdm
from transformers import AutoTokenizer, EsmModel

from interprot.sae_model import SparseAutoencoder
from interprot.utils import get_layer_activations

ESM_DIM = 1280
SAE_DIM = 4096
LAYER = 28

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load ESM model
tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t33_650M_UR50D")
esm_model = EsmModel.from_pretrained("facebook/esm2_t33_650M_UR50D")
esm_model.to(device)
esm_model.eval()

# Load SAE model
checkpoint_path = hf_hub_download(
    repo_id="liambai/InterProt-ESM2-SAEs",
    filename="esm2_plm1280_l28_sae4096.safetensors"
)
sae_model = SparseAutoencoder(ESM_DIM, SAE_DIM)
sae_model.load_state_dict(load_file(checkpoint_path))
sae_model.to(device)
sae_model.eval()

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: pandas 3.0.1
Uninstalling pandas-3.0.1:
  Successfully uninstalled pandas-3.0.1
Found existing installation: polars 1.39.0
Uninstalling polars-1.39.0:
  Successfully uninstalled polars-1.39.0
Found existing installation: interprot 0.1.0
Uninstalling interprot-0.1.0:
  Successfully uninstalled interprot-0.1.0
  Cloning https://github.com/etowahadams/interprot.git to /tmp/pip-req-build-7p1iel6_
  Running command git clone --filter=blob:none --quiet https://github.com/etowahadams/interprot.git /tmp/pip-req-build-7p1iel6_
  Resolved https://github.com/etowahadams/interprot.git to commit e167a57f370426198864532857ce70c261c73530
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached numpy-2.4.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata 

Loading weights:   0%|          | 0/566 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


SparseAutoencoder()

Load the subcellular localization dataset

In [3]:
# Download subcellular localization dataset from https://zenodo.org/records/10631963
!gdown https://drive.google.com/uc?id=1BG91Eu80t546q-9wIDxCaSYpzGh4lPHX

data_path = "balanced.csv"
df = pl.read_csv(data_path)
df = df.filter(pl.col("sequence").str.len_chars() <= 1000)

train_df = df.filter(pl.col("set") == "train")
test_df = df.filter(pl.col("set") != "train")

df.head()

Downloading...
From: https://drive.google.com/uc?id=1BG91Eu80t546q-9wIDxCaSYpzGh4lPHX
To: /content/balanced.csv
100% 6.25M/6.25M [00:00<00:00, 43.4MB/s]


sequence,target,set,validation
str,str,str,str
"""MEVLEEPAPGPGGADAAERRGLRRLLLSGF…","""Cell membrane""","""train""",null
"""MMKTLSSGNCTLNVPAKNSYRMVVLGASRV…","""Cell membrane""","""train""",null
"""MAKRTFSNLETFLIFLLVMMSAITVALLSL…","""Cell membrane""","""train""",null
"""MGNCQAGHNLHLCLAHHPPLVCATLILLLL…","""Cell membrane""","""train""",null
"""MDPSKQGTLNRVENSVYRTAFKLRSVQTLC…","""Cell membrane""","""train""",null


Define some helpers for linear classified training, evaluation, and inspection.

In [4]:
def train_classifier(seq_acts):
    """Trains a linear classifier with a predefined validation split.

    Uses grid search over regularization strengths to find the best classifier.
    """
    X_train = []
    y_train = []
    test_fold = []

    for row in train_df.iter_rows(named=True):
        X_train.append(seq_acts[row['sequence']].cpu().detach().numpy())
        y_train.append(row['target'])
        if row['validation']:
            test_fold.append(0)
        else:
            test_fold.append(-1)

    X_train = np.array(X_train)
    y_train = np.array(y_train)

    classifier = LogisticRegression(multi_class='ovr')
    ps = PredefinedSplit(test_fold=test_fold)
    param_grid = {'C': [0.01, 0.1, 1, 10, 100]}
    grid_search = GridSearchCV(classifier, param_grid, cv=ps, scoring='accuracy')
    grid_search.fit(X_train, y_train)

    best_classifier = grid_search.best_estimator_
    return best_classifier, grid_search.best_params_['C'], grid_search.best_score_


def evaluate_classifier(classifier, seq_acts):
    """Computes the accuracy of the classifier over a test set"""
    X_test = []
    y_test = []
    for row in test_df.iter_rows(named=True):
        X_test.append(seq_acts[row['sequence']].cpu().detach().numpy())
        y_test.append(row['target'])
    X_test = np.array(X_test)
    y_test = np.array(y_test)

    y_pred = classifier.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    return accuracy


def write_sorted_weights(classifier, output_dir):
    """Writes model weights to CSV files in sorted order.

    For multi-class models, writes one CSV per class.
    Each CSV has columns "Index" and "Weight" sorted descending.

    Args:
        output_dir: Directory to write CSV files to
    """
    os.makedirs(output_dir, exist_ok=True)

    for i, class_label in enumerate(classifier.classes_):
        class_label = class_label.replace("/", "_").lower()
        output_file = os.path.join(output_dir, f"class_{class_label}_weights.csv")

        print(f"Class: {class_label}")
        class_weights = classifier.coef_[i]
        sorted_weights = sorted(enumerate(class_weights), key=lambda x: x[1], reverse=True)
        with open(output_file, 'w', newline='') as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(['Index', 'Weight'])
            for index, weight in sorted_weights:
                writer.writerow([index, weight])

        print(pd.read_csv(output_file).head())

## ESM & SAE Inference

Compute and store mean-pooled embeddings for both ESM and SAE

In [5]:
!pip install codecarbon

In [6]:
from codecarbon import EmissionsTracker
import os

#creating folder to save my reslts
os.makedirs("carbon_logs", exist_ok=True)

#creating the main tracker (measures GPU/CPU energy use)
tracker = EmissionsTracker(
    project_name="InterProt_Probe_Activations",
    output_dir="carbon_logs",
    measure_power_secs=15,           #checking power often enough
    log_level="info",                #will show messages in output
    on_csv_write="update"            #re-runs should be fine

)

#start measuring
tracker.start()
print("Codecarbon is now tracking the energy!")

[codecarbon WARNING @ 18:28:57] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 18:28:57] [setup] RAM Tracking...
[codecarbon INFO @ 18:28:57] [setup] CPU Tracking...
[codecarbon WARNING @ 18:28:58] We saw that you have a Intel(R) Xeon(R) CPU @ 2.00GHz but we don't know it. Please contact us.
[codecarbon WARNING @ 18:28:58] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Linux OS detected: Please ensure RAPL files exist, and are readable, at /sys/class/powercap/intel-rapl/subsystem to measure CPU

[codecarbon INFO @ 18:28:58] CPU Model on constant consumption mode: Intel(R) Xeon(R) CPU @ 2.00GHz
[codecarbon WARNING @ 18:28:58] No CPU tracking mode found. Falling back on CPU constant mode.
[codecarbon INFO @ 18:28:58] [setup] GPU Tracking...
[codecarbon INFO @ 18:28:58] Tracking Nvidia GPU via pynvml
[codecarbon INFO @ 18:28:58] The below tracking methods have been set up:
                RAM Tracking Method: RAM po

Codecarbon is now tracking the energy!


In [7]:
seq_esm_acts = {}
seq_sae_acts = {}

for seq_idx, row in tqdm(enumerate(df.iter_rows(named=True)), total=len(df)):
    seq = row['sequence']
    esm_layer_acts = get_layer_activations(
        tokenizer=tokenizer, plm=esm_model, seqs=[seq], layer=LAYER
    )[0][1:-1] # Trim off BoS and EoS tokens

    seq_esm_acts[seq] = torch.mean(esm_layer_acts, axis=0)

    sae_acts = sae_model.get_acts(esm_layer_acts)
    seq_sae_acts[seq] = torch.mean(sae_acts, axis=0)

  0%|          | 40/10461 [00:11<51:48,  3.35it/s][codecarbon INFO @ 18:29:14] Energy consumed for RAM : 0.000042 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 18:29:14] Delta energy consumed for CPU with constant : 0.000177 kWh, power : 42.5 W
[codecarbon INFO @ 18:29:14] Energy consumed for All CPU : 0.000177 kWh
[codecarbon INFO @ 18:29:14] Energy consumed for all GPUs : 0.000235 kWh. Total GPU Power : 56.2984391515711 W
[codecarbon INFO @ 18:29:14] 0.000454 kWh of electricity and 0.000000 L of water were used since the beginning.
  1%|          | 98/10461 [00:26<55:44,  3.10it/s][codecarbon INFO @ 18:29:29] Energy consumed for RAM : 0.000083 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 18:29:29] Delta energy consumed for CPU with constant : 0.000177 kWh, power : 42.5 W
[codecarbon INFO @ 18:29:29] Energy consumed for All CPU : 0.000354 kWh
[codecarbon INFO @ 18:29:29] Energy consumed for all GPUs : 0.000508 kWh. Total GPU Power : 65.55752499334905 W
[codecarbon INFO @ 18:29:29] 0.000945

# Linear classifiers

- Train classifier on ESM embedding (for accuracy comparison)
- Train classifier on SAE embedding
- Inspect and write the weights of the SAE classifier

In [8]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")

    print("Training ESM classifier")
    esm_classifier, _, _ = train_classifier(seq_esm_acts)
    esm_accuracy = evaluate_classifier(esm_classifier, seq_esm_acts)
    print(f"ESM accuracy: {esm_accuracy}")

    print("Training SAE classifier")
    sae_classifier, _, _ = train_classifier(seq_sae_acts)
    sae_accuracy = evaluate_classifier(sae_classifier, seq_sae_acts)
    print(f"SAE accuracy: {sae_accuracy}")
    write_sorted_weights(sae_classifier, 'weights')

Training ESM classifier


[codecarbon INFO @ 19:10:59] Energy consumed for RAM : 0.006993 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 19:10:59] Delta energy consumed for CPU with constant : 0.000177 kWh, power : 42.5 W
[codecarbon INFO @ 19:10:59] Energy consumed for All CPU : 0.029722 kWh
[codecarbon INFO @ 19:10:59] Energy consumed for all GPUs : 0.045936 kWh. Total GPU Power : 48.10512196433605 W
[codecarbon INFO @ 19:10:59] 0.082650 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon INFO @ 19:10:59] 0.004469 g.CO2eq/s mean an estimation of 140.9316709817295 kg.CO2eq/year
[codecarbon INFO @ 19:11:14] Energy consumed for RAM : 0.007034 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 19:11:14] Delta energy consumed for CPU with constant : 0.000177 kWh, power : 42.5 W
[codecarbon INFO @ 19:11:14] Energy consumed for All CPU : 0.029899 kWh
[codecarbon INFO @ 19:11:14] Energy consumed for all GPUs : 0.046072 kWh. Total GPU Power : 32.62342895984711 W
[codecarbon INFO @ 19:11:14] 0.083

ESM accuracy: 0.6312684365781711
Training SAE classifier


[codecarbon INFO @ 19:13:29] Energy consumed for RAM : 0.007409 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 19:13:29] Delta energy consumed for CPU with constant : 0.000177 kWh, power : 42.5 W
[codecarbon INFO @ 19:13:29] Energy consumed for All CPU : 0.031491 kWh
[codecarbon INFO @ 19:13:29] Energy consumed for all GPUs : 0.047312 kWh. Total GPU Power : 33.238497902018764 W
[codecarbon INFO @ 19:13:29] 0.086212 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon INFO @ 19:13:44] Energy consumed for RAM : 0.007450 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 19:13:44] Delta energy consumed for CPU with constant : 0.000177 kWh, power : 42.5 W
[codecarbon INFO @ 19:13:44] Energy consumed for All CPU : 0.031668 kWh
[codecarbon INFO @ 19:13:44] Energy consumed for all GPUs : 0.047452 kWh. Total GPU Power : 33.63828072818242 W
[codecarbon INFO @ 19:13:44] 0.086570 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon INFO @ 19:1

SAE accuracy: 0.6430678466076696
Class: cell membrane
   Index    Weight
0   3815  2.452949
1    550  1.941587
2   2818  1.421556
3   2000  1.230898
4   3154  1.006192
Class: cytoplasm
   Index    Weight
0   3274  1.052673
1   1849  0.988262
2   2966  0.945494
3   2253  0.869542
4   2554  0.816985
Class: endoplasmic reticulum
   Index    Weight
0   1298  1.292515
1   1053  1.252058
2    848  1.205963
3   1055  1.047036
4    565  1.013210
Class: extracellular
   Index    Weight
0   1541  1.284865
1   1470  1.262685
2   1555  1.162771
3   2111  1.112270
4   2472  1.025668
Class: golgi apparatus
   Index    Weight
0   4086  1.568856
1   3729  1.173677
2   1635  1.162739
3    880  1.151911
4     75  1.086463
Class: lysosome_vacuole
   Index    Weight
0   2354  1.229107
1   3592  1.103126
2    598  1.083880
3   3021  1.073236
4   2867  1.073207
Class: mitochondrion
   Index    Weight
0   3277  2.380635
1   1948  1.161483
2   1690  1.129278
3   2745  1.124608
4    326  1.065481
Class: nucleu

In [9]:
#Stopping my tracker!
emissions = tracker.stop()
print(f"Emissions from this training run: {emissions:5f} kg CO2eq")

[codecarbon INFO @ 19:20:54] Energy consumed for RAM : 0.008644 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 19:20:54] Delta energy consumed for CPU with constant : 0.000120 kWh, power : 42.5 W
[codecarbon INFO @ 19:20:54] Energy consumed for All CPU : 0.036742 kWh
[codecarbon INFO @ 19:20:54] Energy consumed for all GPUs : 0.051571 kWh. Total GPU Power : 34.81735160666163 W
[codecarbon INFO @ 19:20:54] 0.096957 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon INFO @ 19:20:54] 0.003356 g.CO2eq/s mean an estimation of 105.8314078289027 kg.CO2eq/year


Emissions from this training run: 0.013453 kg CO2eq
